# CAFA 6 - Exploratory Data Analysis

This notebook explores the competition dataset:
- Protein sequences (train + test)
- GO term annotations
- Ontology structure
- Taxonomy distribution
- Information accretion weights

In [ ]:
# === Google Colab Setup (skip if running locally) ===
import os
if 'COLAB_GPU' in os.environ or 'COLAB_RELEASE_TAG' in os.environ:
    !git clone https://github.com/AyushSharma173/CafaChallenge.git
    %cd CafaChallenge
    !pip install -q obonet biopython h5py lightgbm scikit-learn matplotlib seaborn pyyaml
    !pip install -q kaggle
    # Upload your kaggle.json: click the folder icon -> upload -> kaggle.json
    # Or run: from google.colab import files; files.upload()
    os.makedirs('/root/.kaggle', exist_ok=True)
    if os.path.exists('kaggle.json'):
        !cp kaggle.json /root/.kaggle/ && chmod 600 /root/.kaggle/kaggle.json
    !kaggle competitions download -c cafa-6-protein-function-prediction -p data/raw
    !unzip -qo data/raw/cafa-6-protein-function-prediction.zip -d data/raw/
    %cd notebooks
    print('Colab setup complete!')

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from pathlib import Path

from cafa6.data_loader import load_fasta, load_train_terms, load_taxonomy, load_ia_weights
from cafa6.go_utils import load_go_graph, get_ontology_terms, ASPECT_TO_NAMESPACE, ASPECT_FULL_NAME

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

DATA_DIR = Path('../data/raw')

## 1. Load All Data

In [ ]:
# Load sequences
train_seqs = load_fasta(DATA_DIR / 'Train' / 'train_sequences.fasta')
test_seqs = load_fasta(DATA_DIR / 'Test' / 'testsuperset.fasta')
print(f'Training proteins: {len(train_seqs):,}')
print(f'Test proteins:     {len(test_seqs):,}')

# Load annotations
terms_df = load_train_terms(DATA_DIR / 'Train' / 'train_terms.tsv')
print(f'\nAnnotations: {len(terms_df):,} rows')
print(f'Unique proteins with annotations: {terms_df["EntryID"].nunique():,}')
print(f'Unique GO terms: {terms_df["term"].nunique():,}')
print(f'Aspect values: {terms_df["aspect"].unique()}')

# Load taxonomy
taxonomy_df = load_taxonomy(DATA_DIR / 'Train' / 'train_taxonomy.tsv')
print(f'\nTaxonomy entries: {len(taxonomy_df):,}')
print(f'Unique species: {taxonomy_df["taxonomyID"].nunique():,}')

# Load GO graph
go_graph = load_go_graph(DATA_DIR / 'Train' / 'go-basic.obo')
print(f'\nGO graph: {go_graph.number_of_nodes():,} terms, {go_graph.number_of_edges():,} edges')

# Load IA weights
ia_weights = load_ia_weights(DATA_DIR / 'IA.tsv')
print(f'IA weights for {len(ia_weights):,} terms')

## 2. Sequence Length Analysis

In [ ]:
train_lengths = [len(s) for s in train_seqs.values()]
test_lengths = [len(s) for s in test_seqs.values()]

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Histogram
axes[0].hist(train_lengths, bins=100, alpha=0.7, label='Train', color='steelblue')
axes[0].hist(test_lengths, bins=100, alpha=0.7, label='Test', color='coral')
axes[0].set_xlabel('Sequence Length')
axes[0].set_ylabel('Count')
axes[0].set_title('Sequence Length Distribution')
axes[0].legend()
axes[0].set_xlim(0, 3000)

# Stats
stats = pd.DataFrame({
    'Train': pd.Series(train_lengths).describe(),
    'Test': pd.Series(test_lengths).describe()
})
axes[1].axis('off')
table = axes[1].table(cellText=stats.round(1).values,
                       rowLabels=stats.index,
                       colLabels=stats.columns,
                       loc='center')
table.auto_set_font_size(False)
table.set_fontsize(12)
axes[1].set_title('Sequence Length Statistics')

plt.tight_layout()
plt.show()

# ESM-2 truncation impact
ESM_MAX = 1022
train_over = sum(1 for l in train_lengths if l > ESM_MAX)
test_over = sum(1 for l in test_lengths if l > ESM_MAX)
print(f'\nSequences exceeding ESM-2 max ({ESM_MAX} tokens):')
print(f'  Train: {train_over:,} ({100*train_over/len(train_lengths):.1f}%)')
print(f'  Test:  {test_over:,} ({100*test_over/len(test_lengths):.1f}%)')

## 3. GO Term Analysis

In [ ]:
# Per-ontology stats
for aspect in ['P', 'F', 'C']:
    aspect_df = terms_df[terms_df['aspect'] == aspect]
    n_terms = aspect_df['term'].nunique()
    n_proteins = aspect_df['EntryID'].nunique()
    avg_terms = aspect_df.groupby('EntryID').size().mean()
    print(f'{ASPECT_FULL_NAME[aspect]}: {n_terms:,} unique terms, '
          f'{n_proteins:,} proteins, {avg_terms:.1f} avg terms/protein')

print()

# Term frequency distribution (power law)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, aspect in enumerate(['P', 'F', 'C']):
    aspect_df = terms_df[terms_df['aspect'] == aspect]
    term_counts = aspect_df['term'].value_counts()
    
    axes[idx].loglog(range(1, len(term_counts)+1), term_counts.values, '.', alpha=0.5)
    axes[idx].set_xlabel('Term Rank (log)')
    axes[idx].set_ylabel('Frequency (log)')
    axes[idx].set_title(f'{ASPECT_FULL_NAME[aspect]} - Term Frequency')
    axes[idx].axhline(y=50, color='red', linestyle='--', alpha=0.5)
    n_above = (term_counts >= 50).sum()
    axes[idx].legend([f'min_count=50: {n_above} terms'])

plt.tight_layout()
plt.show()

In [ ]:
# Annotations per protein
annotations_per_protein = terms_df.groupby('EntryID').size()

fig, ax = plt.subplots(figsize=(12, 5))
ax.hist(annotations_per_protein, bins=100, color='steelblue', alpha=0.7)
ax.set_xlabel('Number of GO Annotations')
ax.set_ylabel('Number of Proteins')
ax.set_title('Annotations per Protein')
ax.axvline(annotations_per_protein.median(), color='red', linestyle='--', 
           label=f'Median: {annotations_per_protein.median():.0f}')
ax.legend()
plt.tight_layout()
plt.show()

print(f'Annotations per protein: min={annotations_per_protein.min()}, '
      f'max={annotations_per_protein.max()}, median={annotations_per_protein.median():.0f}')

## 4. GO Ontology Structure

In [ ]:
# Ontology statistics
for aspect_code, namespace in ASPECT_TO_NAMESPACE.items():
    ont_terms = get_ontology_terms(go_graph, aspect_code)
    subgraph = go_graph.subgraph(ont_terms)
    print(f'{ASPECT_FULL_NAME[aspect_code]}:')
    print(f'  Terms: {len(ont_terms):,}')
    print(f'  Edges: {subgraph.number_of_edges():,}')
    
    # Roots (no outgoing edges = no parents)
    roots = [n for n in ont_terms if go_graph.out_degree(n) == 0 and n in go_graph]
    print(f'  Roots: {len(roots)}')
    for r in roots[:3]:
        name = go_graph.nodes[r].get('name', 'unknown')
        print(f'    {r}: {name}')
    print()

## 5. Taxonomy Distribution

In [ ]:
# Top species in training set
species_counts = taxonomy_df['taxonomyID'].value_counts()

fig, ax = plt.subplots(figsize=(14, 6))
top_n = 20
species_counts.head(top_n).plot(kind='bar', ax=ax, color='steelblue')
ax.set_xlabel('Taxonomy ID')
ax.set_ylabel('Number of Proteins')
ax.set_title(f'Top {top_n} Species by Protein Count')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

print(f'Total species: {species_counts.shape[0]:,}')
print(f'Top 5 species cover {species_counts.head(5).sum()/len(taxonomy_df)*100:.1f}% of proteins')

# Species with few proteins
print(f'Species with <10 proteins: {(species_counts < 10).sum():,}')
print(f'Species with only 1 protein: {(species_counts == 1).sum():,}')

## 6. Information Accretion Weights

In [ ]:
ia_values = list(ia_weights.values())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(ia_values, bins=100, color='steelblue', alpha=0.7)
axes[0].set_xlabel('IA Weight')
axes[0].set_ylabel('Count')
axes[0].set_title('Distribution of Information Accretion Weights')

# IA vs term frequency
term_freq = terms_df['term'].value_counts()
common_terms = set(ia_weights.keys()) & set(term_freq.index)
ia_for_plot = [ia_weights[t] for t in common_terms]
freq_for_plot = [term_freq[t] for t in common_terms]

axes[1].scatter(freq_for_plot, ia_for_plot, alpha=0.1, s=5)
axes[1].set_xscale('log')
axes[1].set_xlabel('Term Frequency (log)')
axes[1].set_ylabel('IA Weight')
axes[1].set_title('IA Weight vs Term Frequency')

plt.tight_layout()
plt.show()

print(f'IA weights: min={min(ia_values):.3f}, max={max(ia_values):.3f}, '
      f'mean={np.mean(ia_values):.3f}, median={np.median(ia_values):.3f}')

## 7. Train vs Test Comparison

In [ ]:
# Amino acid composition comparison
def aa_composition(sequences):
    counts = Counter()
    total = 0
    for seq in sequences.values():
        counts.update(seq)
        total += len(seq)
    return {aa: count/total for aa, count in counts.items()}

train_aa = aa_composition(train_seqs)
test_aa = aa_composition(test_seqs)

aa_list = sorted(set(train_aa.keys()) | set(test_aa.keys()))
aa_df = pd.DataFrame({
    'Train': [train_aa.get(aa, 0) for aa in aa_list],
    'Test': [test_aa.get(aa, 0) for aa in aa_list]
}, index=aa_list)

fig, ax = plt.subplots(figsize=(14, 5))
aa_df.plot(kind='bar', ax=ax, width=0.8)
ax.set_xlabel('Amino Acid')
ax.set_ylabel('Frequency')
ax.set_title('Amino Acid Composition: Train vs Test')
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.show()

# Overlap analysis
train_ids = set(train_seqs.keys())
test_ids = set(test_seqs.keys())
overlap = train_ids & test_ids
print(f'Protein ID overlap between train and test: {len(overlap)}')

## Summary

Key takeaways from EDA (fill in after running):
- Number of training/test proteins
- Typical sequence lengths and ESM truncation impact
- GO term distribution follows power law (many rare terms)
- Number of usable terms after min_count filtering
- Species distribution is highly skewed
- Train/test amino acid composition similarity